# Task 2.2 — Albero di decisione costruito a mano

**Obiettivo del Task 2** (consegna): *definire manualmente uno o due classificatori sul file
`manuale.csv`, adottando uno o due modelli illustrati a lezione, e valutarne le prestazioni sullo
stesso file. Dopo aver illustrato i passi per adattare i modelli ai dati, implementare i
classificatori in Python, utilizzando eventualmente delle API.*

Questo notebook copre il **secondo** dei due classificatori richiesti (siamo un gruppo di due
componenti); il primo, il **Naive Bayes**, e' in `02.1_naive_bayes.ipynb`.

Il modello scelto e' l'**albero di decisione** costruito con il criterio dell'*information gain*
(algoritmo ID3). I passi sono quelli visti a lezione:

1. calcolare l'entropia del nodo corrente;
2. per ogni feature, calcolare di quanto l'entropia si riduce dividendo i dati (*information gain*);
3. scegliere la feature con il guadagno massimo e ripetere sui due rami;
4. fermarsi quando un nodo e' puro.

Le motivazioni discorsive sono in `documentation/02.2_albero_decisione.md`; qui c'e' il codice e i
numeri che lo verificano.

In [1]:
import numpy as np
import pandas as pd

SEME = 42          # coerente con il Task 1

manuale = pd.read_csv("../data/manuale.csv").set_index("USERID")
y = manuale["ABBANDONO"]
X = manuale.drop(columns="ABBANDONO")

print("campioni:", len(manuale), " feature:", X.shape[1])
print("distribuzione delle classi:", y.value_counts().to_dict())
manuale

campioni: 12  feature: 8
distribuzione delle classi: {0: 6, 1: 6}


,n_azioni,n_attivita_distinte,n_giorni_attivi,durata_giorni,feature0_media,feature1_media,feature2_media,feature3_media,ABBANDONO
USERID,,,,,,,,,
3969,44,26,5,4.941,-0.167,0.258,-0.053,-0.013,0
635,138,73,7,26.158,0.081,0.228,-0.093,-0.044,0
4344,5,3,1,0.000,-0.320,-0.436,0.107,-0.067,1
1714,97,40,11,27.117,0.014,-0.200,-0.012,-0.057,0
1123,8,5,1,0.013,-0.320,-0.118,0.044,-0.067,1
3637,6,5,1,0.082,-0.320,-0.012,0.023,-0.067,1
4413,94,19,5,5.156,-0.191,-0.409,0.096,-0.031,1
2535,65,42,6,25.987,1.215,-0.436,0.045,0.831,0
4200,7,5,2,3.618,-0.320,0.291,0.035,-0.067,1


## 1. Entropia del nodo radice

L'entropia misura l'incertezza di un nodo: vale 0 se tutti i campioni appartengono alla stessa
classe, e 1 bit se le due classi sono in perfetto equilibrio.

$$H(S) = -\sum_{c} p_c \log_2 p_c$$

`manuale.csv` contiene 6 abbandoni e 6 non-abbandoni per costruzione (il campionamento del Task 1
era stratificato), quindi ci aspettiamo esattamente **1 bit**: la situazione di massima incertezza,
il punto di partenza peggiore possibile.

In [2]:
def entropia(etichette):
    """Entropia di Shannon di un vettore di etichette, misurata in bit."""
    if len(etichette) == 0:
        return 0.0
    p = etichette.value_counts(normalize=True).to_numpy()
    return float(-(p * np.log2(p)).sum()) + 0.0      # il + 0.0 evita il -0.0 sui nodi puri

H_radice = entropia(y)
print(f"entropia della radice = {H_radice:.4f} bit")

entropia della radice = 1.0000 bit


## 2. Come si divide una feature continua

A lezione l'albero divide i dati sui **valori** di una feature categorica: un ramo per ogni valore.
Le nostre otto feature sono pero' tutte **continue** (conteggi e medie), quindi un ramo per valore
darebbe un albero con un campione per foglia — inutile.

Si usa allora la regola standard per gli attributi numerici: si ordinano i valori presenti e si
prova a tagliare a meta' fra due valori consecutivi distinti, `feature <= soglia`. I punti medi
sono gli **unici** tagli interessanti, perche' due soglie fra la stessa coppia di valori producono
esattamente la stessa partizione dei dati: con *n* valori distinti bastano *n-1* prove.

E' la stessa scelta che fa scikit-learn: `DecisionTreeClassifier` non accetta affatto variabili
categoriche, sa fare solo split su soglia numerica.

In [3]:
def soglie_candidate(colonna):
    """Punti medi fra valori consecutivi distinti: gli unici tagli che cambiano la partizione."""
    v = np.unique(colonna)
    return (v[:-1] + v[1:]) / 2

print("n_giorni_attivi   valori distinti:", np.unique(X["n_giorni_attivi"]))
print("                  soglie da provare:", soglie_candidate(X["n_giorni_attivi"]))
print()
print("numero di soglie da provare per ciascuna feature:")
for f in X.columns:
    print(f"  {f:22s} {len(soglie_candidate(X[f])):2d}")

n_giorni_attivi   valori distinti: [ 1  2  5  6  7  9 11]
                  soglie da provare: [ 1.5  3.5  5.5  6.5  8.  10. ]

numero di soglie da provare per ciascuna feature:
  n_azioni               11
  n_attivita_distinte     9
  n_giorni_attivi         6
  durata_giorni          11
  feature0_media          8
  feature1_media         10
  feature2_media         11
  feature3_media          7


## 3. Information gain di ogni feature

L'*information gain* di uno split e' la riduzione di entropia che produce, pesata sulla numerosita'
dei due rami:

$$IG(S, f, t) = H(S) - \frac{|S_{\le t}|}{|S|} H(S_{\le t}) - \frac{|S_{> t}|}{|S|} H(S_{> t})$$

Per ogni feature proviamo tutte le sue soglie candidate e teniamo la migliore, poi ordiniamo le
otto feature per guadagno decrescente.

In [4]:
def guadagno(X, y, feature, soglia):
    """Information gain dello split 'feature <= soglia' sul nodo (X, y)."""
    sinistra = y[X[feature] <= soglia]
    destra   = y[X[feature] >  soglia]
    peso = len(sinistra) / len(y)
    return entropia(y) - peso * entropia(sinistra) - (1 - peso) * entropia(destra)

def classifica_feature(X, y):
    """Per ogni feature la soglia migliore e il guadagno che ottiene."""
    righe = []
    for f in X.columns:
        s, g = max(((s, guadagno(X, y, f, s)) for s in soglie_candidate(X[f])), key=lambda t: t[1])
        righe.append({"feature": f, "soglia_migliore": float(s), "information_gain": g})
    return pd.DataFrame(righe).sort_values("information_gain", ascending=False).reset_index(drop=True)

classifica_feature(X, y).round(4)          # la soglia esatta resta quella non arrotondata

,feature,soglia_migliore,information_gain
0,n_attivita_distinte,22.500,1.0000
1,n_azioni,31.500,0.6549
2,n_giorni_attivi,3.500,0.6549
3,durata_giorni,4.813,0.6549
4,feature0_media,-0.179,0.6549
5,feature3_media,-0.062,0.6549
6,feature2_media,-0.039,0.4591
7,feature1_media,0.082,0.1957


**Osservazione.** `n_attivita_distinte` ottiene un information gain di **1.0000**, cioe' il massimo
possibile: azzera l'intera entropia della radice in un colpo solo. Significa che la soglia trovata
separa i 12 campioni **senza un solo errore**, e che entrambi i rami sono foglie pure.

Le sei feature successive si equivalgono quasi tutte (IG 0.6549): non e' un caso, e nel Task 1
avevamo gia' visto perche'. `n_azioni`, `n_giorni_attivi`, `durata_giorni` e in parte
`feature0_media` sono tutte misure della **lunghezza della storia** dello studente, quindi
descrivono la stessa cosa e producono partizioni molto simili. Ci torniamo al punto 7.

In [5]:
radice = classifica_feature(X, y).iloc[0]
FEATURE, SOGLIA = radice["feature"], radice["soglia_migliore"]

sinistra = manuale[X[FEATURE] <= SOGLIA].sort_values(FEATURE)
destra   = manuale[X[FEATURE] >  SOGLIA].sort_values(FEATURE)

print(f"split scelto: {FEATURE} <= {SOGLIA:.3f}\n")
print(f"ramo SI  ({len(sinistra):2d} campioni): {FEATURE} = {list(sinistra[FEATURE])}")
print(f"         etichette = {list(sinistra['ABBANDONO'])}  ->  entropia {entropia(sinistra['ABBANDONO']):.4f}")
print(f"ramo NO  ({len(destra):2d} campioni): {FEATURE} = {list(destra[FEATURE])}")
print(f"         etichette = {list(destra['ABBANDONO'])}  ->  entropia {entropia(destra['ABBANDONO']):.4f}")

split scelto: n_attivita_distinte <= 22.500

ramo SI  ( 6 campioni): n_attivita_distinte = [3, 5, 5, 5, 9, 19]
         etichette = [1, 1, 1, 1, 1, 1]  ->  entropia 0.0000
ramo NO  ( 6 campioni): n_attivita_distinte = [26, 40, 42, 59, 62, 73]
         etichette = [0, 0, 0, 0, 0, 0]  ->  entropia 0.0000


## 4. L'albero risultante

Entrambi i rami hanno entropia 0, quindi l'algoritmo si ferma subito: **non c'e' un secondo
livello da costruire**. L'albero e' un unico nodo di decisione con due foglie (in letteratura si
chiama *decision stump*):

```
n_attivita_distinte <= 22.5 ?
├── si  -> ABBANDONO = 1   (6 campioni, tutti abbandoni)
└── no  -> ABBANDONO = 0   (6 campioni, tutti non-abbandoni)
```

In parole: **uno studente che ha toccato al massimo 22 attivita' diverse del corso viene
classificato come abbandono.**

Non abbiamo potato l'albero ne' fermato la crescita in anticipo: e' l'algoritmo stesso a fermarsi,
perche' un nodo puro non ha piu' entropia da ridurre. Al punto 8 discutiamo perche' un risultato
cosi' netto sia un campanello d'allarme e non un successo.

In [6]:
def cresci(X, y, profondita=0, max_profondita=3):
    """ID3 con soglie numeriche: sceglie ricorsivamente lo split con guadagno massimo."""
    if entropia(y) == 0 or profondita == max_profondita:
        return {"foglia": int(y.mode().iloc[0]), "n": len(y)}
    c = classifica_feature(X, y).iloc[0]
    if c["information_gain"] <= 1e-12:
        return {"foglia": int(y.mode().iloc[0]), "n": len(y)}
    maschera = X[c["feature"]] <= c["soglia_migliore"]
    return {"feature": c["feature"], "soglia": c["soglia_migliore"], "ig": c["information_gain"], "n": len(y),
            "si": cresci(X[maschera],  y[maschera],  profondita + 1, max_profondita),
            "no": cresci(X[~maschera], y[~maschera], profondita + 1, max_profondita)}

def stampa(nodo, indent=""):
    if "foglia" in nodo:
        print(f"{indent}=> ABBANDONO = {nodo['foglia']}   ({nodo['n']} campioni)")
        return
    print(f"{indent}{nodo['feature']} <= {nodo['soglia']:.3f} ?   IG={nodo['ig']:.4f}   ({nodo['n']} campioni)")
    print(f"{indent}  si:"); stampa(nodo["si"], indent + "     ")
    print(f"{indent}  no:"); stampa(nodo["no"], indent + "     ")

def predici(nodo, X):
    def scendi(riga):
        n = nodo
        while "foglia" not in n:
            n = n["si"] if riga[n["feature"]] <= n["soglia"] else n["no"]
        return n["foglia"]
    return X.apply(scendi, axis=1)

albero = cresci(X, y)
stampa(albero)

n_attivita_distinte <= 22.500 ?   IG=1.0000   (12 campioni)
  si:
     => ABBANDONO = 1   (6 campioni)
  no:
     => ABBANDONO = 0   (6 campioni)


## 5. Prestazioni sul file `manuale.csv`

La consegna chiede di valutare il classificatore **sullo stesso file** su cui e' stato costruito.
E' una valutazione *in-sample*: dice se il modello ha imparato i dati che ha visto, non se
generalizza. Al punto 8 lo mettiamo alla prova sui 7.035 studenti di `training.csv`.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pred = predici(albero, X)

risultati = manuale.assign(previsto=pred, corretto=lambda d: d["ABBANDONO"] == d["previsto"])
print(risultati[[FEATURE, "ABBANDONO", "previsto", "corretto"]].sort_values(FEATURE).to_string())

print(f"\naccuratezza = {accuracy_score(y, pred):.4f}")
print(f"precisione  = {precision_score(y, pred):.4f}")
print(f"richiamo    = {recall_score(y, pred):.4f}")
print(f"F1          = {f1_score(y, pred):.4f}")
print("\nmatrice di confusione [righe = vero, colonne = previsto]:")
print(pd.DataFrame(confusion_matrix(y, pred), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))

        n_attivita_distinte  ABBANDONO  previsto  corretto
USERID                                                    
4344                      3          1         1      True
1123                      5          1         1      True
3637                      5          1         1      True
4200                      5          1         1      True
3071                      9          1         1      True
4413                     19          1         1      True
3969                     26          0         0      True
1714                     40          0         0      True
2535                     42          0         0      True
3058                     59          0         0      True
5557                     62          0         0      True
635                      73          0         0      True

accuratezza = 1.0000
precisione  = 1.0000
richiamo    = 1.0000
F1          = 1.0000

matrice di confusione [righe = vero, colonne = previsto]:
        prev 0  prev 1
vero 0 

## 6. Verifica con le API di scikit-learn

La consegna chiede di implementare il classificatore *"utilizzando eventualmente delle API"*.
Avendolo gia' scritto a mano, usiamo l'API come **controprova**: se il nostro codice implementa
davvero ID3, `DecisionTreeClassifier(criterion="entropy")` deve scegliere la stessa feature, la
stessa soglia e produrre le stesse 12 predizioni.

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text

albero_sk = DecisionTreeClassifier(criterion="entropy", random_state=SEME).fit(X, y)
print(export_text(albero_sk, feature_names=list(X.columns)))

pred_sk = albero_sk.predict(X)
print("stessa feature alla radice:", X.columns[albero_sk.tree_.feature[0]] == FEATURE)
print("stessa soglia            :", bool(np.isclose(albero_sk.tree_.threshold[0], SOGLIA)))
print("stesse 12 predizioni     :", bool((pred_sk == pred.to_numpy()).all()))
print("numero di foglie         :", albero_sk.get_n_leaves(), "(come il nostro albero)")

|--- n_attivita_distinte <= 22.50
|   |--- class: 1
|--- n_attivita_distinte >  22.50
|   |--- class: 0

stessa feature alla radice: True
stessa soglia            : True
stesse 12 predizioni     : True
numero di foglie         : 2 (come il nostro albero)


## 7. Controprova: e se togliessimo la feature dominante?

Con un solo nodo l'albero non mostra la parte ricorsiva dell'algoritmo, e soprattutto resta il
dubbio che tutto dipenda da quella singola feature.

Dal Task 1 sappiamo pero' che le prime sei feature sono **misure alternative della stessa
quantita'** — quanto a lungo lo studente e' rimasto attivo — e che `n_giorni_attivi` e
`durata_giorni` sono correlate fra loro a -0,554 con il target. La domanda ha quindi senso
statistico, non e' un esercizio di stile: *se il proxy migliore non fosse disponibile, l'albero
troverebbe la stessa struttura usando gli altri?*

Rimuoviamo `n_attivita_distinte` e rieseguiamo lo stesso identico algoritmo.

In [9]:
X_ridotto = X.drop(columns=FEATURE)

print("classifica delle feature alla radice, senza", FEATURE, ":")
print(classifica_feature(X_ridotto, y).round(4).to_string(index=False))

albero_ridotto = cresci(X_ridotto, y)
print("\nalbero ottenuto:")
stampa(albero_ridotto)

pred_ridotto = predici(albero_ridotto, X_ridotto)
print(f"\naccuratezza sul file manuale = {accuracy_score(y, pred_ridotto):.4f}")

# quante feature pareggiano allo split successivo, dove i campioni sono solo 7?
maschera = X_ridotto[albero_ridotto["feature"]] > albero_ridotto["soglia"]
figlio, y_figlio = X_ridotto[maschera], y[maschera]
print(f"\nclassifica nel nodo di destra ({len(figlio)} campioni), dove avviene il secondo split:")
print(classifica_feature(figlio, y_figlio).round(4).to_string(index=False))

for nome, cl in [("radice", classifica_feature(X_ridotto, y)), ("nodo di destra", classifica_feature(figlio, y_figlio))]:
    massimo = cl["information_gain"].max()
    pari = cl[np.isclose(cl["information_gain"], massimo)]["feature"].tolist()
    print(f"\n{nome}: guadagno massimo {massimo:.4f}, ottenuto da {len(pari)} feature -> {pari}")

classifica delle feature alla radice, senza n_attivita_distinte :
        feature  soglia_migliore  information_gain
       n_azioni           31.500            0.6549
n_giorni_attivi            3.500            0.6549
  durata_giorni            4.813            0.6549
 feature0_media           -0.179            0.6549
 feature3_media           -0.062            0.6549
 feature2_media           -0.039            0.4591
 feature1_media            0.082            0.1957

albero ottenuto:
n_azioni <= 31.500 ?   IG=0.6549   (12 campioni)
  si:
     => ABBANDONO = 1   (5 campioni)
  no:
     feature0_media <= -0.179 ?   IG=0.5917   (7 campioni)
       si:
          => ABBANDONO = 1   (1 campioni)
       no:
          => ABBANDONO = 0   (6 campioni)

accuratezza sul file manuale = 1.0000

classifica nel nodo di destra (7 campioni), dove avviene il secondo split:
        feature  soglia_migliore  information_gain
 feature0_media          -0.1790            0.5917
 feature2_media           0.


radice: guadagno massimo 0.6549, ottenuto da 5 feature -> ['n_azioni', 'n_giorni_attivi', 'durata_giorni', 'feature0_media', 'feature3_media']

nodo di destra: guadagno massimo 0.5917, ottenuto da 2 feature -> ['feature0_media', 'feature2_media']


**Esito.** Senza la feature dominante l'albero cresce su **due livelli** e raggiunge comunque 12
campioni su 12: la struttura non dipende da quale proxy usiamo, come previsto. La regola diventa
*"meno di 32 azioni, oppure `feature0_media` sotto -0,179"*, che e' la stessa idea detta con altre
due variabili.

Va segnalata pero' una fragilita' che il conteggio finale rende esplicita: **il criterio non
riesce quasi mai a scegliere**. Alla radice cinque feature su sette ottengono esattamente lo
stesso guadagno (0,6549), e nel nodo di destra due su sette pareggiano a 0,5917. In tutti questi
casi la feature che finisce nell'albero e' semplicemente **la prima nell'ordine delle colonne**,
non la migliore: con 12 campioni alla radice e 7 nel nodo figlio, partizioni diverse producono
la stessa riduzione di entropia e l'information gain non ha risoluzione sufficiente per
distinguerle.

E' il sintomo concreto del problema che misuriamo adesso: non e' che l'albero abbia scelto male,
e' che con dodici campioni **non c'e' abbastanza informazione per scegliere**.

## 8. Anticipazione del Task 4: quanto vale davvero questo albero?

Dodici campioni sono pochissimi, e un information gain di 1.0000 non significa che il problema sia
facile: significa che con 12 punti e 8 feature continue e' quasi sempre **possibile** trovare un
taglio che separa tutto. La domanda vera e' se quella soglia descrive gli studenti o solo questi
dodici.

Applichiamo l'albero, senza rieseguire nulla, ai 7.035 studenti di `training.csv` (che non
contiene i 12 campioni di `manuale.csv`).

In [10]:
training = pd.read_csv("../data/training.csv").set_index("USERID")
y_tr = training["ABBANDONO"]
X_tr = training.drop(columns="ABBANDONO")

pred_tr = predici(albero, X_tr)
print(f"studenti valutati: {len(y_tr)}  (positivi {y_tr.mean():.1%})")
print(f"accuratezza = {accuracy_score(y_tr, pred_tr):.4f}   F1 = {f1_score(y_tr, pred_tr):.4f}")
print("\nmatrice di confusione:")
print(pd.DataFrame(confusion_matrix(y_tr, pred_tr), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))

# la soglia scelta su 12 campioni sarebbe stata la stessa con 7.035?
scan = pd.DataFrame({"soglia": np.arange(2, 61)})
scan["accuratezza"] = [accuracy_score(y_tr, (X_tr[FEATURE] <= s).astype(int)) for s in scan["soglia"]]
migliore = scan.loc[scan["accuratezza"].idxmax()]
print(f"\nsoglia scelta a mano su 12 campioni : {SOGLIA}   -> accuratezza {accuracy_score(y_tr, pred_tr):.4f}")
print(f"soglia ottima stimata su 7.035      : {migliore['soglia']:.0f}   -> accuratezza {migliore['accuratezza']:.4f}")

studenti valutati: 7035  (positivi 57.7%)
accuratezza = 0.7734   F1 = 0.8032

matrice di confusione:
        prev 0  prev 1
vero 0    2189     786
vero 1     808    3252



soglia scelta a mano su 12 campioni : 22.5   -> accuratezza 0.7734
soglia ottima stimata su 7.035      : 28   -> accuratezza 0.7808


## 9. Analisi critica

**Il divario fra i due numeri e' il risultato piu' importante del Task 2.** L'albero passa da
**100%** su `manuale.csv` a circa **77%** su `training.csv`: la differenza e' il costo di aver
scelto sia la feature sia la soglia guardando gli stessi 12 campioni su cui poi ci siamo misurati.
E' un caso da manuale di valutazione *in-sample*, e giustifica da solo perche' nel Task 5 si separi
il test set prima di toccare i dati.

**Cio' che invece regge.** La soglia trovata a mano (22,5) e quella ottima stimata su 7.035
studenti sono vicinissime, e l'accuratezza che producono differisce di meno di un punto. Il taglio
non e' un artefatto: la relazione "poche attivita' distinte -> abbandono" esiste davvero nei dati.
Cio' che i 12 campioni sovrastimano non e' *dove* tagliare, ma **quanto pulito** sia il taglio.

**Il limite del modello, non della stima.** Un solo nodo su una sola feature e' un modello a
capacita' bassissima: qualunque studente con piu' di 22 attivita' distinte riceve la stessa
risposta, indipendentemente da tutto il resto. Il 77% e' il tetto di questa forma, non un errore
di addestramento. Nel Task 5 vedremo di quanto lo superano i modelli con piu' capacita'.

**Il caveat ereditato dal Task 1.** Va ripetuto anche qui: la regola "poche attivita' -> abbandono"
e' in parte **tautologica**, perche' un abbandono e' per definizione la fine dell'attivita'. La
soglia 22,5 non spiega *perche'* uno studente abbandoni, registra che chi ha abbandonato ha fatto
meno cose. E' una predizione utile solo se la si applica a una finestra iniziale di osservazione,
non all'intera storia — esattamente la verifica fatta nell'appendice del Task 1.